In [1]:
import os
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from astra.src.embedding import AstraEmbedding
from astra.src.preprocessing import contrastive_data_loader

2025-11-11 11:39:44.333239: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-11 11:39:44.433869: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-11 11:39:45.983530: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Data Loader for Contrastive Learning (Anchor, Positive, Negative)

In [2]:
#
# This script demonstrates how to use the contrastive_data_loader function
#
n_views = 3
batch_size = 200
apply_white_noise=[False, True, True]
apply_binning=[False, False, True]
apply_outlier=[False, False, True]
maxlens=[{'g': 100, 'r': 200, 'i': 50}, {'g': 100, 'r': 150, 'i': 50}, {'g': 100, 'r': 200, 'i': 50}] 
bin_widths = [5, 5, 5]
drop_rates = [0.0, 0.0, 0.60]
noise_levels = [0.10, 0.0, 0.10] 
buffer_size = 10000  
build_seq_len=tf.cast(sum(maxlens[0].values()), tf.int32)  # Final fixed length for sequences 

path_to_read = "/media3/majumder/dataset/resampled_multi-class/val/"
#
loaders = contrastive_data_loader( source=path_to_read,
                                    n_views=n_views, 
                                    seed=np.random.randint(1024), 
                                    batch_size=batch_size,
                                    build_seq_len=build_seq_len,
                                    apply_white_noise=apply_white_noise,
                                    noise_levels=noise_levels,
                                    apply_binning=apply_binning,
                                    apply_outlier=apply_outlier,
                                    maxlens=maxlens,
                                    bin_widths=bin_widths,
                                    drop_rates=drop_rates,
                                    buffer_size=buffer_size
                                )

# im1, im2, im3 = next(iter(loaders))
# print(im1, im2, im3)
# for anchor, positive, negative in loaders:
#   print(anchor.keys(), anchor.values(), positive, negative)
#   break

# pbar_train = tqdm(loaders)
# for batch in pbar_train:
#     tf.print(batch[0]['input'].shape, batch[0]['times'].shape, batch[0]['band_info'].shape, batch[0]['mask'].shape)
#     tf.print(batch[1]['input'].shape, batch[1]['times'].shape, batch[1]['band_info'].shape, batch[1]['mask'].shape)
#     tf.print(batch[2]['input'].shape, batch[2]['times'].shape, batch[2]['band_info'].shape, batch[2]['mask'].shape)
#     tf.print(batch[1]['mask'][1, 297: 303])
#     tf.print(batch[1]['times'][1, 297: 303,:])
#     break

2025-11-11 11:39:48.746567: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-11 11:39:48.826166: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Searching for TFRecord files using pattern: /media3/majumder/dataset/resampled_multi-class/val/*/chunk_*.record


Found 27 TFRecord files.


Example filenames:


- /media3/majumder/dataset/resampled_multi-class/val/BCEP/chunk_0.record




In [12]:
strategy = tf.distribute.get_strategy()
distributed_dataset = strategy.experimental_distribute_dataset(loaders)
pbar_train = tqdm(distributed_dataset)
for batch in pbar_train:
    tf.print(batch[0]['input'].shape, batch[0]['times'].shape, batch[0]['band_info'].shape, batch[0]['mask'].shape)
    tf.print(batch[1]['input'].shape, batch[1]['times'].shape, batch[1]['band_info'].shape, batch[1]['mask'].shape)
    tf.print(batch[2]['input'].shape, batch[2]['times'].shape, batch[2]['band_info'].shape, batch[2]['mask'].shape)
    tf.print(batch[1]['mask'][1, 297: 303], batch[0]['mask'][1, 297: 303], batch[2]['mask'][1, 297: 303])
    tf.print(batch[1]['times'][1, 297: 303,:] , batch[0]['times'][1, 297: 303,:], batch[2]['times'][1, 297: 303,:])
    break

0it [00:00, ?it/s]

TensorShape([200, 350, 1]) TensorShape([200, 350, 1]) TensorShape([200, 350, 1]) TensorShape([200, 350])
TensorShape([200, 350, 1]) TensorShape([200, 350, 1]) TensorShape([200, 350, 1]) TensorShape([200, 350])
TensorShape([200, 350, 1]) TensorShape([200, 350, 1]) TensorShape([200, 350, 1]) TensorShape([200, 350])
[0 0 0 1 1 1] [0 0 0 0 0 0] [0 0 0 1 0 0]
[[1158.26501]
 [1164.36365]
 [1165.26538]
 [0]
 [0]
 [0]] [[474.190765]
 [474.191193]
 [474.19165]
 [387.348175]
 [743.51947]
 [743.519958]] [[1727.46143]
 [1727.46338]
 [1727.46533]
 [0]
 [743.51947]
 [743.519958]]


2025-11-11 11:47:34.466655: W tensorflow/core/kernels/data/cache_dataset_ops.cc:917] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
0it [00:01, ?it/s]


In [ ]:
for anchor, positive, negative in loaders:
  print(anchor, positive, negative)
  break

### Generate Time-series embeddings for dart

In [ ]:
# Define dimensions
d_model = 512

In [ ]:
emb = AstraEmbedding(d_model=d_model, base=10000, rate=0.1, use_band_info=True, use_drop=True, mjd=True)
tensor = emb(negative['input'], negative['times'], negative['band_info'])
print(tensor.shape)
print(tensor)